In [ ]:
# =============================================================================
# WSRP - WORKFORCE SCHEDULING AND ROUTING PROBLEM
# MVP Orchestration: Base Model (M0 - TSP)
# =============================================================================

# 1. Importing internal modules from our Clean Architecture (src/)
from src.utils.logger import project_logger
from src.core.data_generator import generate_wsrp_m0_instance
from src.models.model_0 import build_model_m0
from src.solvers.engine import solve_model
from src.utils.parsers import print_routes

def run_m0_experiment():
    """
    Orchestrates the end-to-end execution of the M0 Baseline Model.
    """
    project_logger.info("Initializing WSRP M0 Optimization Pipeline...")

    # ---------------------------------------------------------
    # PHASE 1: Data Ingestion (I/O Isolation)
    # ---------------------------------------------------------
    project_logger.info("PHASE 1: Generating synthetic instance (5 properties)...")
    data_payload = generate_wsrp_m0_instance(num_properties=5, random_seed=42)
    project_logger.info(f"Data generated successfully. Total nodes mapped: {data_payload['num_nodes']}")

    # ---------------------------------------------------------
    # PHASE 2: Mathematical Formulation
    # ---------------------------------------------------------
    project_logger.info("PHASE 2: Building the MILP Polyhedron (Pyomo) with MTZ constraints...")
    abstract_model = build_model_m0(data_payload)

    # ---------------------------------------------------------
    # PHASE 3: Inference Engine (Strategy Pattern)
    # ---------------------------------------------------------
    project_logger.info("PHASE 3: Invoking Gurobi Solver seeking absolute optimality...")

    # We pass the abstract model to our engine. MIPGap 0.0 guarantees mathematical proof.
    solved_model, metrics = solve_model(
        model=abstract_model,
        solver_name='gurobi_direct',
        time_limit=300,
        mip_gap=0.0
    )

    # ---------------------------------------------------------
    # PHASE 4: Results & Parsing
    # ---------------------------------------------------------
    if metrics['solver_status'] == 'ok':
        project_logger.info(f"Optimization finished in {metrics['cpu_time_seconds']} seconds.")
        project_logger.info("Translating binary variables into chronological routes...")
        print_routes(solved_model)
    else:
        project_logger.error(f"Solver failed. Termination condition: {metrics['termination_condition']}")

# Execute the pipeline
if __name__ == "__main__":
    run_m0_experiment()
